# Этап 1. Профилирование `yield_df.csv`: подсчёт дефектов

Блокнот только **измеряет** дефекты, ничего не исправляет (очистка делается в `cleaning.py`).
Номера дефектов и ожидаемые значения взяты из `README.md` (раздел 3). Дефекты 6 и 7 относятся к сырым файлам `temp.csv` и `rainfall.csv`, которые в работе не используются, поэтому здесь не измеряются.
В конце есть сводная таблица «ожидали / получили».

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data/yield_df.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

KEY = ["Area", "Item", "Year"]
TARGET = "hg/ha_yield"
RAIN = "average_rain_fall_mm_per_year"
PEST = "pesticides_tonnes"
TEMP = "avg_temp"

results = {}  # сюда складываем найденные числа для итоговых таблиц

## 1. Обзор таблицы

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns="Unnamed: 0", errors="ignore")  # служебный индекс

print("Размер:", df.shape)
print("\nТипы столбцов:")
print(df.dtypes)
print("\nПропусков всего:", int(df.isna().sum().sum()))
print("Полных дубликатов строк (без служебного индекса):", int(df.duplicated().sum()))
print("\nСтран:", df["Area"].nunique(), "| культур:", df["Item"].nunique(),
      "| лет:", df["Year"].nunique(), f"({df['Year'].min()}-{df['Year'].max()})")

results["rows"] = len(df)
results["areas"] = df["Area"].nunique()
results["items"] = df["Item"].nunique()
results["years"] = df["Year"].nunique()
results["missing"] = int(df.isna().sum().sum())
results["full_duplicates"] = int(df.duplicated().sum())

df.describe().T

**Замечание.** README говорит об отсутствии полностью совпадающих строк. Это верно только вместе со служебным индексом `Unnamed: 0`. После его удаления полные дубликаты есть: это частный случай дефекта 1 (повтор ключа при одинаковой температуре).

## 2. Дефект 1: повтор ключа (`Area`, `Item`, `Year`)
Одна запись об урожайности не должна встречаться дважды. Проверяем, сколько строк делят ключ и чем отличаются.

In [ ]:
dup_mask = df.duplicated(KEY, keep=False)
n_unique_keys = len(df.drop_duplicates(KEY))

# Для каждого повторяющегося ключа: сколько разных значений урожайности и температуры
g = (df[dup_mask]
     .groupby(KEY)
     .agg(n_rows=(TARGET, "size"),
          n_yield=(TARGET, "nunique"),
          n_temp=(TEMP, "nunique"),
          temp_gap=(TEMP, lambda s: s.max() - s.min())))
conf_temp = g[g["n_temp"] > 1]
conf_yield = g[g["n_yield"] > 1]

results.update(
    rows_sharing_key=int(dup_mask.sum()),
    share_sharing_key=round(dup_mask.mean() * 100, 1),
    unique_keys=n_unique_keys,
    extra_rows=len(df) - n_unique_keys,
    dup_keys=len(g),
    keys_temp_differs=len(conf_temp),
    keys_yield_differs=len(conf_yield),
    median_temp_gap=round(conf_temp["temp_gap"].median(), 2),
)

print(f"Строк с общим ключом: {dup_mask.sum()} ({dup_mask.mean():.1%})")
print(f"Уникальных ключей: {n_unique_keys}")
print(f"Лишних строк: {len(df) - n_unique_keys}")
print("\nПовторяющихся ключей:", len(g))
print("  из них с разной температурой:", len(conf_temp))
print("  из них с одинаковой температурой (точные дубликаты):", int((g["n_temp"] == 1).sum()))
print("Ключей с разной урожайностью:", len(conf_yield))
print("Медианный разрыв температуры, °C:", results["median_temp_gap"])

In [ ]:
# Пример: один ключ, одна урожайность, разные температуры
example_key = conf_temp.sort_values("temp_gap", ascending=False).index[0]
df[(df[KEY] == pd.Series(example_key, index=KEY)).all(axis=1)]

## 3. Дефект 2: «осадки» не меняются по годам

In [ ]:
rain_per_area = df.groupby("Area")[RAIN].nunique()
results["areas_rain_varies"] = int((rain_per_area > 1).sum())

print("Стран, у которых осадки меняются по годам:", results["areas_rain_varies"], "из", len(rain_per_area))
print("Стран с ровно одним значением осадков за все годы:", int((rain_per_area == 1).sum()))

# Для сравнения: сколько разных значений за годы внутри страны у других признаков
within_area = pd.DataFrame({
    "rain": rain_per_area,
    "temp": df.groupby("Area")[TEMP].nunique(),
    "pest": df.groupby("Area")[PEST].nunique(),
})
within_area.describe().loc[["mean", "min", "max"]]

## 4. Дефект 3: несогласованность температуры внутри (страна, год)
Температура должна быть одной на страну и год, независимо от культуры.

In [ ]:
temp_per_ay = (df.groupby(["Area", "Year"])[TEMP]
                 .agg(n_values="nunique", gap=lambda s: s.max() - s.min()))
bad_ay = temp_per_ay[temp_per_ay["n_values"] > 1]

results.update(
    area_year_total=len(temp_per_ay),
    area_year_multi_temp=len(bad_ay),
    median_gap_area_year=round(bad_ay["gap"].median(), 2),
)

print(f"Сочетаний (страна, год): {len(temp_per_ay)}")
print(f"С более чем одним значением температуры: {len(bad_ay)} ({len(bad_ay) / len(temp_per_ay):.1%})")
print(f"Разброс max - min среди них: медиана {results['median_gap_area_year']} °C, максимум {bad_ay['gap'].max():.2f} °C")

## 5. Дефекты 4 и 5: асимметрия и выбросы

In [ ]:
results["skew_yield"] = round(df[TARGET].skew(), 2)
results["skew_pest"] = round(df[PEST].skew(), 2)

print("Асимметрия урожайности:", results["skew_yield"])
print("Асимметрия урожайности после log1p:", round(np.log1p(df[TARGET]).skew(), 2))
print("Асимметрия пестицидов:", results["skew_pest"])
print("Асимметрия пестицидов после log1p:", round(np.log1p(df[PEST]).skew(), 2))

In [ ]:
def iqr_outlier_share(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - k * iqr) | (s > q3 + k * iqr)).mean()

outliers = (df.groupby("Item")[TARGET]
              .apply(iqr_outlier_share)
              .rename("outlier_share")
              .sort_values(ascending=False))

results["outlier_wheat"] = round(outliers["Wheat"] * 100, 1)
results["outlier_plantains"] = round(outliers["Plantains and others"] * 100, 1)
results["outlier_maize"] = round(outliers["Maize"] * 100, 1)

# Для сравнения: доля выбросов по IQR на всей таблице (без разделения по культурам)
print("Доля выбросов по IQR на всей таблице:", f"{iqr_outlier_share(df[TARGET]):.1%}")
print("\nДоля выбросов по IQR внутри культуры:")
(outliers * 100).round(1).to_frame("outlier_%")

## 6. Особенности выборки (не дефект данных)
Смотрим, насколько неравномерно распределены строки по странам и годам. От этого зависят метрики на всей выборке и размеры фолдов при `GroupKFold` по странам.

In [ ]:
rows_by_area = df["Area"].value_counts()

results["rows_india"] = int(rows_by_area["India"])
results["share_india"] = round(rows_by_area["India"] / len(df) * 100, 1)
results["rows_montenegro"] = int(rows_by_area["Montenegro"])

def with_share(s):
    return s.to_frame("rows").assign(share_pct=(s / len(df) * 100).round(1))

print("5 стран с наибольшим числом строк:")
print(with_share(rows_by_area.head(5)))
print("\n5 стран с наименьшим числом строк:")
print(with_share(rows_by_area.tail(5)))
print(f"\nДоля пяти крупнейших стран: {rows_by_area.head(5).sum() / len(df):.1%}")
print(f"Строк на страну: медиана {rows_by_area.median():.0f}, min {rows_by_area.min()}, max {rows_by_area.max()}")
print(f"Отношение max/min: {rows_by_area.max() / rows_by_area.min():.0f}")

# Годы: в диапазоне 1990-2013 не хватает одного года, поэтому лет 23, а не 24
missing_years = sorted(set(range(df["Year"].min(), df["Year"].max() + 1)) - set(df["Year"].unique()))
print("\nГоды, отсутствующие внутри диапазона:", missing_years)

## 7. Графики
Все рисунки сохраняются в папку `figures/` для отчёта.

In [ ]:
def save_show(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()

# 7.1 Гистограмма урожайности: исходная и после log1p
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df[TARGET], bins=60)
ax[0].set_title(f"Урожайность, hg/ha (асимметрия {results['skew_yield']})")
ax[0].set_xlabel("hg/ha_yield"); ax[0].set_ylabel("число строк")

y_log = np.log1p(df[TARGET])
ax[1].hist(y_log, bins=60)
ax[1].set_title(f"log1p(урожайность) (асимметрия {y_log.skew():.2f})")
ax[1].set_xlabel("log1p(hg/ha_yield)")
fig.tight_layout()
save_show(fig, "01_yield_hist.png")

In [ ]:
# 7.2 Boxplot урожайности по культурам (усы = правило IQR с k=1.5, точки за усами = выбросы)
order = df.groupby("Item")[TARGET].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([df.loc[df["Item"] == i, TARGET] for i in order])
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("hg/ha_yield")
ax.set_title("Урожайность по культурам")
fig.tight_layout()
save_show(fig, "02_yield_boxplot_by_item.png")

In [ ]:
# 7.3 Строки по странам: 15 крупнейших и распределение числа строк на страну
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
top = rows_by_area.head(15).sort_values()
ax[0].barh(top.index, top.values)
ax[0].set_title("15 стран с наибольшим числом строк")
ax[0].set_xlabel("число строк")

ax[1].hist(rows_by_area.values, bins=30)
ax[1].set_title("Распределение числа строк на страну")
ax[1].set_xlabel("строк на страну"); ax[1].set_ylabel("число стран")
fig.tight_layout()
save_show(fig, "03_rows_by_area.png")

In [ ]:
# 7.4 Разброс температуры внутри одной страны и одного года (должен быть 0), только ненулевые
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(bad_ay["gap"], bins=40)
ax.set_title("Разброс температуры внутри (страна, год), только ненулевые")
ax.set_xlabel("max - min, °C"); ax.set_ylabel("число пар (страна, год)")
fig.tight_layout()
save_show(fig, "04_temp_gap_within_area_year.png")

In [ ]:
# 7.5 Корреляции числовых признаков (Пирсон и Спирмен)
num_cols = ["Year", TARGET, RAIN, PEST, TEMP]
short = {"Year": "Year", TARGET: "yield", RAIN: "rain", PEST: "pest", TEMP: "temp"}
labels = [short[c] for c in num_cols]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = df[num_cols].corr(method=method)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
    ax.set_title(method.capitalize())
fig.colorbar(im, ax=axes, shrink=0.8)
save_show(fig, "05_correlations.png")

**Как читать корреляции.** `rain` постоянна внутри страны (дефект 2), поэтому её связь с урожайностью отражает различия **между странами**, а не влияние осадков по годам. Корреляция Спирмена устойчивее к скошенным распределениям (дефекты 4 и 5), чем Пирсона.

## 8. Сводная таблица дефектов: масштаб, влияние на модель, правило проверки
Нумерация совпадает с `README.md`. Перекос выборки (раздел 6) в таблицу не включён: это не дефект данных.

In [ ]:
r = results
defects = pd.DataFrame([
    {"№": 1, "Дефект": "Размножение строк при объединении: повтор ключа (Area, Item, Year)",
     "Масштаб": f"{r['rows_sharing_key']} строк ({r['share_sharing_key']} %) делят ключ; лишних строк {r['extra_rows']}; "
                f"{r['keys_temp_differs']} ключей с разной температурой (медианный разрыв {r['median_temp_gap']} °C)",
     "Влияние на модель": "Копии одной записи утяжеляют часть объектов при обучении; для ключей с разной температурой признак зашумлён. "
                          "При разбиении по странам копии попадают в одну часть, поэтому утечки между обучением и тестом нет.",
     "Правило проверки": "уникальность ключа"},
    {"№": 2, "Дефект": "Осадки постоянны внутри страны",
     "Масштаб": f"у {r['areas'] - r['areas_rain_varies']} из {r['areas']} стран одно значение за все годы",
     "Влияние на модель": "Признак несёт только различия между странами, а не изменение по годам. "
                          "Его нельзя читать как годовой климатический фактор, это нужно учитывать при анализе важности признаков.",
     "Правило проверки": "константность"},
    {"№": 3, "Дефект": "Несколько значений температуры у одной пары (страна, год)",
     "Масштаб": f"{r['area_year_multi_temp']} из {r['area_year_total']} пар ({r['area_year_multi_temp'] / r['area_year_total']:.1%}); "
                f"медианный разброс {r['median_gap_area_year']} °C",
     "Влияние на модель": "Шум в признаке: один и тот же объект описан разными температурами. Ограничивает достижимую точность.",
     "Правило проверки": "согласованность температуры"},
    {"№": 4, "Дефект": "Асимметрия целевой переменной и выбросы",
     "Масштаб": f"асимметрия {r['skew_yield']}; выбросы по IQR: пшеница {r['outlier_wheat']} %, "
                f"плантаны {r['outlier_plantains']} %, кукуруза {r['outlier_maize']} %",
     "Влияние на модель": "RMSE определяется редкими большими значениями. Ridge хуже работает на скошенной цели; "
                          "возможна замена на log1p с обратным преобразованием.",
     "Правило проверки": "выбросы и асимметрия"},
    {"№": 5, "Дефект": "Скошенное распределение пестицидов",
     "Масштаб": f"асимметрия {r['skew_pest']}",
     "Влияние на модель": "Влияет на Ridge (масштаб признака); деревья нечувствительны. Возможен log1p внутри Pipeline.",
     "Правило проверки": "выбросы и асимметрия"},
])

pd.set_option("display.max_colwidth", None)
defects.to_csv(FIG_DIR / "defects_table.csv", index=False, encoding="utf-8-sig")
defects

## Сводка: ожидали / получили

In [ ]:
expected = {
    "rows": 28242, "areas": 101, "items": 10, "years": 23, "missing": 0,
    "rows_sharing_key": 18949, "share_sharing_key": 67.1, "unique_keys": 13130, "extra_rows": 15112,
    "keys_temp_differs": 3690, "keys_yield_differs": 0, "median_temp_gap": 4.49,
    "areas_rain_varies": 0, "area_year_total": 2250, "area_year_multi_temp": 575,
    "skew_yield": 1.79, "skew_pest": 3.49,
    "outlier_wheat": 8.6, "outlier_plantains": 7.6, "outlier_maize": 5.0,
    "rows_india": 4048, "share_india": 14.3, "rows_montenegro": 24,
    "full_duplicates": 2310,
}
summary = pd.DataFrame({
    "ожидали (README)": pd.Series(expected),
    "получили": pd.Series(results).reindex(list(expected)),
})
summary["совпало"] = (summary["ожидали (README)"] - summary["получили"]).abs() < 0.05
summary